# Flood Simulation — Trier, Germany (14-Day HQ100 Event)

Simulates a complete 14-day riverine flood event with **hourly time steps** (336
frames total).  The flood extent is approximated by progressively eroding the
HQ100 polygon inward — see
`utils/flood_interpolation.py` for the algorithm.

| Phase | Hours | Days | Flood level |
|---|---|---|---|
| Pre-event | 0–47 | 1–2 | 0 % (no flooding) |
| Rising water | 48–143 | 3–6 | 0 % → 100 % |
| HQ100 peak | 144–191 | 7–8 | 100 % (full flood) |
| Receding | 192–287 | 9–12 | 100 % → 0 % |
| Post-event | 288–335 | 13–14 | 0 % (no flooding) |

Intermediate flood geometries are cached in
`data/processed/flood_interpolation/` and reused on subsequent runs.

In addition to the flood polygon itself, the animation tracks **hospitals,
fire stations, power stations and water stations**, and now also the
**NCNN power/water connections** between hospitals/fire stations and their
nearest infrastructure (reusing the NCNN routes already computed in
`system_overview.ipynb`, see `utils/ncnn.py`). A facility becomes **dead**
when either its own geometry is covered by the flood polygon, or the
power/water station it is connected to becomes dead — modelling cascading
infrastructure failure, not just direct flood damage. All of this logic
lives in `utils/flood_status.py`; the notebook only orchestrates and
visualizes it.

**Keyboard shortcuts** (once the simulation is displayed):
`Space`/`K` — play/pause ·
`←`/`→` — step 1 h ·
`↑`/`↓` — step 1 d

In [2]:
from __future__ import annotations

import os
from pathlib import Path

import geopandas as gpd
import osmnx as ox
import pandas as pd
from shapely.ops import unary_union
from IPython.display import HTML, display

from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_hazard_data_path,
    get_roa_outputs_path,
    load_or_fetch_osm_features,
    set_notbook_wd,
)
from css_geodata_service.robustness_of_accessibility.utils.flood_interpolation import (
    SIMULATION_HOURS,
    _RISE_START, _RISE_END, _PEAK_END, _FALL_END,
    build_flood_animation_html,
    compute_hourly_flood_progress,
    load_or_compute_flood_stages,
)
from css_geodata_service.robustness_of_accessibility.utils.flood_status import (
    compute_flood_status_by_stage,
    compute_dependency_status_by_stage,
)
from css_geodata_service.robustness_of_accessibility.utils.ncnn import (
    load_or_calculate_ncnn_routes,
)

ox.settings.log_console = False
ox.settings.use_cache = True

print(f"osmnx     : {ox.__version__}")
print(f"geopandas : {gpd.__version__}")

osmnx     : 1.9.3
geopandas : 1.1.3


## 1. Configuration & Paths

In [3]:
set_notbook_wd()

place_name: str = RoaNotebookConfig.place_name   # "Trier, Germany"
event           = RoaNotebookConfig.event         # HQ100

cache_dir:       Path = get_roa_cache_path()
output_dir:      Path = get_roa_outputs_path()
hazard_data_path: Path = get_roa_hazard_data_path(event=event)

output_dir.mkdir(parents=True, exist_ok=True)

print(f"Place         : {place_name}")
print(f"Hazard event  : {event}")
print(f"Hazard file   : {hazard_data_path}")
print(f"Cache dir     : {cache_dir}")
print(f"Output dir    : {output_dir}")

Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Place         : Trier, Germany
Hazard event  : M
Hazard file   : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\input\Flooding\HazardAreas\nz_hazardArea_fluival_M-DE_cropped_trier.geojson
Cache dir     : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\processed
Output dir    : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output


## 2. Study-Area Boundary

Loads the pre-processed administrative boundary of Trier from cache
(generated by `system_overview.ipynb`).  Falls back to OSM if the cache
file is absent.

In [4]:
boundary_cache = cache_dir / f"services/boundary_geom_{place_name}.geojson"

if boundary_cache.exists():
    print("Loading boundary from cache …")
    boundary_gdf  = gpd.read_file(boundary_cache)
    boundary_geom = unary_union(boundary_gdf.geometry)
else:
    print("Fetching boundary from OSM …")
    place_gdf     = ox.geocode_to_gdf(place_name)
    boundary_geom = unary_union(place_gdf.geometry)
    gpd.GeoDataFrame(geometry=[boundary_geom], crs="EPSG:4326").to_file(
        boundary_cache, driver="GeoJSON"
    )

minx, miny, maxx, maxy = boundary_geom.bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

print(f"Boundary loaded — centroid ({center_lat:.4f}°N, {center_lon:.4f}°E)")

Loading boundary from cache …
Boundary loaded — centroid (49.7779°N, 6.6495°E)


## 3. HQ100 Flood Data & Stage Interpolation

The `load_or_compute_flood_stages` utility erodes the HQ100 flood polygon
inward in discrete steps to approximate intermediate water levels.  50
stages are pre-computed once and cached as a single GeoJSON file;
subsequent runs skip this step entirely.

Increase `n_stages` (e.g. to 100) for finer transitions at the cost of a
slightly longer first-run computation time.

In [5]:
if not hazard_data_path.exists():
    raise FileNotFoundError(
        f"HQ100 flood data not found:\n  {hazard_data_path}\n"
        "Download the hazard data and place it under "
        "data/input/Flooding/HazardAreas/ as described in the prerequisites."
    )

hq100_gdf = gpd.read_file(hazard_data_path)
print(f"HQ100 source polygons loaded : {len(hq100_gdf)}")

# 50 stages give smooth visual transitions.
# Raise n_stages for finer granularity; lower it for faster first-run.
N_STAGES = 50

stages = load_or_compute_flood_stages(
    cache_dir=cache_dir,
    hq100_gdf=hq100_gdf,
    n_stages=N_STAGES,
    place_name=place_name,
)

non_empty = sum(1 for s in stages if s["geojson"] is not None)
print(f"Flood stages ready           : {len(stages)} total, {non_empty} with polygon")

HQ100 source polygons loaded : 108
Flood stages ready           : 50 total, 49 with polygon


## 4. Facilities & Infrastructure — Direct Flood Status

Loads the same POI / infrastructure feature types already investigated in
`system_overview.ipynb` — hospitals, fire stations, power infrastructure
(substations + plants) and water infrastructure (water works + water towers
+ pumping stations) — from the shared cache (or fetches them from OSM on
first run).

For every pre-computed flood stage, each facility is checked
**independently** against that stage's flood polygon: a facility is
*directly flooded* when its own geometry intersects the flood polygon —
nothing more. Road accessibility is not used anywhere in this notebook.

This section only computes **direct** flood status. Section 5 below builds
on top of it — using the NCNN power/water connections — to determine when a
hospital or fire station also becomes unavailable *indirectly*, because its
connected infrastructure is flooded.

The flood-status logic itself lives in `utils/flood_status.py`
(`compute_flood_status_by_stage`) — this notebook only calls it and passes
the result on to the visualization. It does not implement the check itself.

In [6]:
services_cache_dir = cache_dir / "services"
services_cache_dir.mkdir(parents=True, exist_ok=True)

hospitals = load_or_fetch_osm_features(
    services_cache_dir / f"hospitals_{place_name}.geojson",
    boundary_geom, {"amenity": ["hospital"]}
)
fire_stations = load_or_fetch_osm_features(
    services_cache_dir / f"fire_stations_{place_name}.geojson",
    boundary_geom, {"amenity": ["fire_station"]}
)
power_substations = load_or_fetch_osm_features(
    services_cache_dir / f"power_substations_{place_name}.geojson",
    boundary_geom, {"power": ["substation"]}
)
power_plants = load_or_fetch_osm_features(
    services_cache_dir / f"power_plants_{place_name}.geojson",
    boundary_geom, {"power": ["plant"]}
)
water_towers = load_or_fetch_osm_features(
    services_cache_dir / f"water_towers_{place_name}.geojson",
    boundary_geom, {"man_made": ["water_tower"]}
)
water_works = load_or_fetch_osm_features(
    services_cache_dir / f"water_works_{place_name}.geojson",
    boundary_geom, {"man_made": ["water_works"]}
)
pumping_stations = load_or_fetch_osm_features(
    services_cache_dir / f"pumping_stations_{place_name}.geojson",
    boundary_geom, {"man_made": ["pumping_station"]}
)

# Power and water infrastructure are combined into single categories for this
# visualization — the same grouping already used for NCNN in system_overview.ipynb.
power_stations = gpd.GeoDataFrame(
    pd.concat([power_substations, power_plants], ignore_index=True), crs="EPSG:4326"
)
water_stations = gpd.GeoDataFrame(
    pd.concat([water_works, water_towers, pumping_stations], ignore_index=True), crs="EPSG:4326"
)

facility_gdfs = {
    "hospital":     hospitals,
    "fire_station": fire_stations,
    "power":        power_stations,
    "water":        water_stations,
}

print(f"Hospitals      : {len(hospitals)}")
print(f"Fire stations  : {len(fire_stations)}")
print(f"Power stations : {len(power_stations)}  (substations + plants)")
print(f"Water stations : {len(water_stations)}  (works + towers + pumping stations)")

Hospitals      : 3
Fire stations  : 12
Power stations : 131  (substations + plants)
Water stations : 11  (works + towers + pumping stations)


In [7]:
facility_flood_status = compute_flood_status_by_stage(facility_gdfs, stages)

# Sanity check: nothing should be flooded at the pre-event stage (progress 0),
# and the peak-flood stage should have the highest (or equal) flooded count.
pre_event_stage_idx = min(range(len(stages)), key=lambda i: stages[i]["progress"])
peak_stage_idx = max(range(len(stages)), key=lambda i: stages[i]["progress"])

print(f"{'Facility type':14s}  {'Total':>5s}  {'Flooded @ pre-event':>20s}  {'Flooded @ peak':>15s}")
print("-" * 62)
for ftype, gdf in facility_gdfs.items():
    total = len(gdf)
    flooded_pre = sum(facility_flood_status[ftype][pre_event_stage_idx])
    flooded_peak = sum(facility_flood_status[ftype][peak_stage_idx])
    print(f"{ftype:14s}  {total:5d}  {flooded_pre:20d}  {flooded_peak:15d}")

Facility type   Total   Flooded @ pre-event   Flooded @ peak
--------------------------------------------------------------
hospital            3                     0                3
fire_station       12                     0                4
power             131                     0               32
water              11                     0                0


## 5. NCNN Infrastructure Connections & Cascading Failure

Hospitals and fire stations depend on power and water infrastructure to
function. This section determines when a POI becomes unavailable **not**
because it is itself flooded, but because the power or water station it
relies on is.

The POI → infrastructure assignment comes from the **Network-Constrained
Nearest-Neighbor (NCNN)** routes already computed in `system_overview.ipynb`
(`utils/ncnn.py`, `load_or_calculate_ncnn_routes`) — the nearest power/water
station reachable via the road network from each hospital/fire station.
Since this notebook builds `power_stations` / `water_stations` with the
exact same feature sets and ordering used there, calling
`load_or_calculate_ncnn_routes` again simply reuses the already-cached NCNN
result — it is not recomputed.

**Cascading rule** (implemented in `utils/flood_status.py`,
`compute_dependency_status_by_stage` — not in this notebook):

A hospital or fire station is **dead** at a given stage when *either*:
1. its own geometry is directly covered by the flood polygon (section 4), **or**
2. the power station connected to it (via NCNN) is dead at that stage, **or**
3. the water station connected to it (via NCNN) is dead at that stage.

A power/water station's own dead status is still purely direct-flood-based
(section 4) — this section does not introduce any further cascading beyond
POI ← infrastructure (e.g. no dependency *between* power and water
stations, and no road-accessibility reasoning).

In [8]:
network_cache = cache_dir / f"network/drive_graph_{place_name}.graphml"

if network_cache.exists():
    print("Loading road network from cache …")
    road_network = ox.load_graphml(network_cache)
else:
    print("Downloading road network from OSM (may take ~1–2 min) …")
    road_network = ox.graph_from_polygon(polygon=boundary_geom, network_type=RoaNotebookConfig.network_type)
    network_cache.parent.mkdir(parents=True, exist_ok=True)
    ox.save_graphml(road_network, filepath=network_cache)

# Undirected so one-way restrictions don't block underground infrastructure
# paths (power/water pipes are bidirectional by nature) — same choice made
# for the NCNN computation in system_overview.ipynb.
road_network_undirected = road_network.to_undirected()

print(f"Road network nodes : {road_network.number_of_nodes():,}")
print(f"Road network edges : {road_network.number_of_edges():,}")

Loading road network from cache …
Road network nodes : 5,817
Road network edges : 12,687


In [9]:
poi_gdfs = {
    "hospital":     hospitals,
    "fire_station": fire_stations,
}
infrastructure_gdfs = {
    "power": power_stations,
    "water": water_stations,
}

print("Computing / loading NCNN routes (POI → nearest infrastructure via road network) …")
ncnn_results = load_or_calculate_ncnn_routes(
    cache_dir=cache_dir,
    poi_gdfs=poi_gdfs,
    infrastructure_gdfs=infrastructure_gdfs,
    street_network=road_network_undirected,
    place_name=place_name,
)

print(f"{'POI type':15s}  {'Infra type':10s}  {'POIs':>4s}  {'Mean (m)':>9s}  {'Max (m)':>8s}")
print("-" * 57)
for poi_type, infra_dict in ncnn_results.items():
    for infra_type, gdf in infra_dict.items():
        finite = gdf[gdf["route_length_m"] < float("inf")]["route_length_m"]
        mean_m = f"{finite.mean():.0f}" if len(finite) else "—"
        max_m  = f"{finite.max():.0f}"  if len(finite) else "—"
        print(f"{poi_type:15s}  {infra_type:10s}  {len(gdf):>4d}  {mean_m:>9s}  {max_m:>8s}")

Computing / loading NCNN routes (POI → nearest infrastructure via road network) …
POI type         Infra type  POIs   Mean (m)   Max (m)
---------------------------------------------------------
hospital         power          3        250       424
hospital         water          3       2186      2614
fire_station     power         12        749      3789
fire_station     water         12       2444      5800


In [10]:
dependency_status = compute_dependency_status_by_stage(
    infrastructure_gdfs=infrastructure_gdfs,
    connections=ncnn_results,
    direct_flooded_by_stage=facility_flood_status,
)

# Sanity check: for hospitals/fire stations, the combined (direct + cascading)
# dead count at peak flood should be >= the direct-only count from section 4 —
# cascading failure can only add dead facilities, never remove them.
print(f"{'POI type':14s}  {'Direct @ peak':>13s}  {'Direct+Cascading @ peak':>24s}")
print("-" * 55)
for poi_type in ncnn_results:
    direct_peak = sum(facility_flood_status[poi_type][peak_stage_idx])
    combined_peak = sum(dependency_status[poi_type]["dead_by_stage"][peak_stage_idx])
    print(f"{poi_type:14s}  {direct_peak:13d}  {combined_peak:24d}")
    assert combined_peak >= direct_peak, "cascading failure must not reduce the dead count"

POI type        Direct @ peak   Direct+Cascading @ peak
-------------------------------------------------------
hospital                    3                         3
fire_station                4                         6


## 6. Simulation Schedule Overview

In [11]:
print("=" * 56)
print("  FLOOD SIMULATION SCHEDULE — 14-day HQ100 event")
print("=" * 56)
print(f"  Total frames  : {SIMULATION_HOURS} (one per hour)")
print(f"  Flood stages  : {len(stages)} pre-computed levels")
print()
print(f"  Phase 1 — Pre-event    : Hours   0–{_RISE_START - 1:3d}  (Days  1–2 )")
print(f"  Phase 2 — Rising water : Hours {_RISE_START:3d}–{_RISE_END  - 1:3d}  (Days  3–6 )")
print(f"  Phase 3 — HQ100 peak   : Hours {_RISE_END:3d}–{_PEAK_END  - 1:3d}  (Days  7–8 )")
print(f"  Phase 4 — Receding     : Hours {_PEAK_END:3d}–{_FALL_END  - 1:3d}  (Days  9–12)")
print(f"  Phase 5 — Post-event   : Hours {_FALL_END:3d}–{SIMULATION_HOURS - 1:3d}  (Days 13–14)")
print("=" * 56)
print()

# Symmetry check
p_h0   = compute_hourly_flood_progress(0)
p_h96  = compute_hourly_flood_progress(96)   # midway through rising phase
p_h168 = compute_hourly_flood_progress(168)  # midway through peak
p_h240 = compute_hourly_flood_progress(240)  # midway through recession
p_h335 = compute_hourly_flood_progress(335)
print("  Symmetry check (rising vs. receding should mirror each other):")
print(f"    Hour   0 (pre)         : {p_h0:.3f}")
print(f"    Hour  96 (mid-rise)    : {p_h96:.3f}")
print(f"    Hour 168 (mid-peak)    : {p_h168:.3f}")
print(f"    Hour 240 (mid-recede)  : {p_h240:.3f}  ← should equal mid-rise")
print(f"    Hour 335 (post)        : {p_h335:.3f}")

  FLOOD SIMULATION SCHEDULE — 14-day HQ100 event
  Total frames  : 336 (one per hour)
  Flood stages  : 50 pre-computed levels

  Phase 1 — Pre-event    : Hours   0– 47  (Days  1–2 )
  Phase 2 — Rising water : Hours  48–143  (Days  3–6 )
  Phase 3 — HQ100 peak   : Hours 144–191  (Days  7–8 )
  Phase 4 — Receding     : Hours 192–287  (Days  9–12)
  Phase 5 — Post-event   : Hours 288–335  (Days 13–14)

  Symmetry check (rising vs. receding should mirror each other):
    Hour   0 (pre)         : 0.000
    Hour  96 (mid-rise)    : 0.500
    Hour 168 (mid-peak)    : 1.000
    Hour 240 (mid-recede)  : 0.500  ← should equal mid-rise
    Hour 335 (post)        : 0.000


## 7. Build & Launch Flood Simulation

Generates a **self-contained HTML animation** and displays it in the
notebook.  All pre-computed flood polygons, facility/infrastructure
markers (with combined direct + cascading dead status), and NCNN
power/water connection lines are embedded directly in the HTML file — no
kernel or server interaction is needed after generation.

The HTML file is also saved to `data/output/flood_simulation.html` and
can be opened directly in any browser for a full-screen presentation.

**Facility visualization:** each facility type is drawn as a small circular
marker in its own colour (red = hospitals, blue = fire stations,
orange = power stations, cyan = water stations). Hospitals and fire
stations grey out when **dead** — either directly flooded or cut off from
a required power/water connection (section 5); power and water stations
grey out only when directly flooded. Availability is fully reversible.

**Connection visualization:** each NCNN power connection is drawn as a
**yellow** line and each water connection as a **dark blue** line, from a
hospital/fire station to its nearest power/water station. A connection
turns **grey** the moment its target station is dead — making it
immediately visible *why* a facility went dark (e.g. its power line goes
grey, then the hospital marker itself greys out on the next check).

A panel in the top-right corner tracks live available/operational counts
for every facility type and connection type as the simulation plays;
hovering a marker or line shows its name and current status.

**In-player controls:**

| Control | Action |
|---|---|
| `► Play` / `⏸ Pause` | Toggle animation |
| `−1h` / `+1h` | Step backward / forward one hour |
| `−1d` / `+1d` | Step backward / forward one day (24 h) |
| **Slider** | Jump to any hour in the simulation |
| **Speed** | 1 / 4 / 8 / 16 / 24 fps |
| `Space` / `K` | Play / Pause (keyboard) |
| `←` / `→` | Step 1 hour (keyboard) |
| `↑` / `↓` | Step 1 day (keyboard) |

In [12]:
FACILITY_STYLE = {
    "hospital":     {"label": "Hospitals",      "color": "#CC2222"},
    "fire_station": {"label": "Fire Stations",  "color": "#1144CC"},
    "power":        {"label": "Power Stations", "color": "#FF8C00"},
    "water":        {"label": "Water Stations", "color": "#00AEEF"},
}

# Hospitals/fire stations use the combined direct+cascading dead status;
# power/water stations only ever have a direct flood status (section 4).
facility_dead_status = {
    **{poi_type: dependency_status[poi_type]["dead_by_stage"] for poi_type in dependency_status},
    "power": facility_flood_status["power"],
    "water": facility_flood_status["water"],
}

facility_layers = {
    ftype: {
        "gdf": facility_gdfs[ftype],
        "flooded_by_stage": facility_dead_status[ftype],
        "label": style["label"],
        "color": style["color"],
    }
    for ftype, style in FACILITY_STYLE.items()
}

CONNECTION_STYLE = {
    "power": {"label": "Power Connections", "color": "#FFD500"},  # yellow
    "water": {"label": "Water Connections", "color": "#00308F"},  # dark blue
}

connection_layers = {
    infra_type: {
        "label": style["label"],
        "color": style["color"],
        "poi_connections": {
            poi_type: {
                "gdf": ncnn_results[poi_type][infra_type],
                "dead_by_stage": dependency_status[poi_type]["connections"][infra_type]["dead_by_stage"],
            }
            for poi_type in dependency_status
        },
    }
    for infra_type, style in CONNECTION_STYLE.items()
}

html_path = build_flood_animation_html(
    boundary_geom=boundary_geom,
    stages=stages,
    output_path=output_dir / "flood_simulation.html",
    center_lat=center_lat,
    center_lon=center_lon,
    facility_layers=facility_layers,
    connection_layers=connection_layers,
)

print(f"Animation file : {html_path}")
print(f"File size      : {html_path.stat().st_size / 1024:.0f} KB")
print()
print("Open the HTML file directly in a browser for a full-screen presentation.")

Animation file : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output\flood_simulation.html
File size      : 577 KB

Open the HTML file directly in a browser for a full-screen presentation.


In [13]:
import base64

html_bytes = html_path.read_bytes()
html_b64 = base64.b64encode(html_bytes).decode("ascii")

display(HTML(
    f'<iframe src="data:text/html;base64,{html_b64}" width="100%" height="780px" '
    f'frameborder="0" '
    f'style="border-radius:8px; box-shadow:0 2px 14px rgba(0,0,0,0.2);">'
    f'</iframe>'
))

print(f"\nFor full-screen use, open directly in a browser:\n  {html_path}")

C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")



For full-screen use, open directly in a browser:
  C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output\flood_simulation.html
